# Commodity Delivery Gap Analysis
## Oil Tanker Imports by US PADD + Derivative Commodities (Helium, NGL)

**Data Sources:**
- **EIA API v2** — Crude oil imports by PADD/port, petroleum movements by transport mode
- **EIA STEO** — Short Term Energy Outlook (forward projections)
- **USGS MCS** — Helium supply/demand (annual, offline here)
- **AISstream.io** — Real-time vessel tracking (tanker positions + ETAs)
- **Yahoo Finance** — Commodity futures (WTI, NatGas, RBOB, Heating Oil)

**Key Thesis:** Helium has no substitute and is extracted as a byproduct of natural gas processing. NatGas import/production disruptions are a leading indicator for helium supply gaps.

**Key terms:**
- **PADD** — Petroleum Administration for Defense District (5 US regions used by EIA for petroleum reporting)
- **MBBL** — thousand barrels (stock levels, import volumes)
- **Bcf** — billion cubic feet (natural gas import volumes)
- **Mcm** — million cubic meters (helium production/demand)
- **Mcf** — thousand cubic feet (helium pricing; 1 Mcm ≈ 35.3 MMcf)
- **Z-score** — standard deviations from trailing average; negative = below normal

### Analytical Framework

This notebook traces a supply-side cascade through public EIA data:

```
Crude Imports by PADD     Are deliveries below normal?
    │                         │
    ├─ Inventory Adequacy    How many days of supply remain?
    │   ├─ Days of Supply    Stocks ÷ daily consumption
    │   ├─ Seasonal Norm     Current vs 5-year average
    │   └─ SPR Buffer        Government reserve capacity
    │
    ├─ NatGas Throughput     Pipeline + LNG import flows
    │   └─ Helium Risk       Co-production depends on NatGas
    │
    ├─ Composite Scorecard   Oil + NatGas z-scores blended
    │   └─ STEO Forward      EIA forecast overlay
    │
    └─ Futures Divergence    Physical gap vs market pricing
```

Each section builds on the previous. The **composite scorecard** synthesizes all signals into a single delivery-risk metric. See the [Market Intelligence Briefing](03_market_intelligence_briefing.ipynb) for trade ideas based on these signals.

In [ ]:
# --- Colab Setup (skip if running locally) ---
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/hb-cam/commodity-flow-intelligence.git
    import os
    from getpass import getpass
    # Optional: enter your EIA API key for live data (press Enter to skip)
    _eia_key = getpass('EIA API Key (Enter to skip): ')
    if _eia_key:
        os.environ['EIA_API_KEY'] = _eia_key
        os.environ['USE_LIVE_API'] = 'true'
    _ais_key = getpass('AISstream API Key (Enter to skip): ')
    if _ais_key:
        os.environ['AISSTREAM_API_KEY'] = _ais_key
    print('Setup complete.' + (' Live mode enabled.' if _eia_key else ' Using offline data.'))

#### Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

from commodity_flow import config, eia, offline, analysis, futures
from commodity_flow.provenance import ProvenanceTracker, DataSource

plt.style.use(config.PLOT_STYLE)
plt.rcParams['figure.figsize'] = config.PLOT_FIGSIZE
plt.rcParams['font.size'] = config.PLOT_FONTSIZE

prov = ProvenanceTracker()

print(f"Live API mode: {config.USE_LIVE_API}")
print(f"EIA API key: {'configured' if config.EIA_API_KEY else 'not set (using offline data)'}")

---
## 1. Load Data

All datasets loaded in a single step. Toggle `USE_LIVE_API` in `.env` to switch between live EIA data and offline mode.

In [ ]:
if config.USE_LIVE_API and config.EIA_API_KEY:
    print("Pulling from EIA API v2...")
    df_imports = eia.fetch_crude_imports_by_padd(config.EIA_API_KEY)
    prov.record(DataSource("Crude imports", "EIA API v2", "petroleum/move/imp", True,
        rows=len(df_imports), date_range=f"{df_imports['date'].min().date()} to {df_imports['date'].max().date()}"))

    df_stocks = eia.fetch_weekly_stocks(config.EIA_API_KEY)
    prov.record(DataSource("Weekly stocks", "EIA API v2", "petroleum/stoc/wstk", True,
        rows=len(df_stocks), date_range=f"{df_stocks['date'].min().date()} to {df_stocks['date'].max().date()}",
        notes="Total petroleum by PADD (crude-only not available at PADD level)"))

    df_steo = eia.fetch_steo_projections(config.EIA_API_KEY)
    prov.record(DataSource("STEO projections", "EIA API v2", "steo", True,
        rows=len(df_steo), notes="CONIPUS (net imports), COPRPUS (production), PAPR_WORLD"))

    df_natgas = eia.fetch_natgas_imports(config.EIA_API_KEY)
    prov.record(DataSource("NatGas imports", "EIA API v2", "natural-gas/move/poe1", True,
        rows=len(df_natgas), date_range=f"{df_natgas['date'].min().date()} to {df_natgas['date'].max().date()}",
        notes="Pipeline + LNG split at national level (Bcf)"))

    print(f"Imports: {len(df_imports)} | Stocks: {len(df_stocks)} | STEO: {len(df_steo)} | NatGas: {len(df_natgas)}")
else:
    print("Using offline data (set USE_LIVE_API=true in .env for real data)")
    df_imports = offline.generate_offline_imports()
    prov.record(DataSource("Crude imports", "Offline", "offline generator", False, rows=len(df_imports)))

    df_stocks = offline.generate_offline_stocks()
    prov.record(DataSource("Weekly stocks", "Offline", "offline generator", False, rows=len(df_stocks)))

    df_steo = offline.generate_offline_steo()
    prov.record(DataSource("STEO projections", "Offline", "offline generator", False, rows=len(df_steo)))

    df_natgas = offline.generate_offline_natgas_imports()
    prov.record(DataSource("NatGas imports", "Offline", "offline generator", False, rows=len(df_natgas),
        notes="US is now net LNG exporter; offline LNG imports are near-zero"))

# Always offline (annual PDF sources, no API)
df_helium = offline.generate_offline_helium()
prov.record(DataSource("Helium supply/demand", "Offline (USGS MCS)", "offline generator", False,
    rows=len(df_helium), notes="USGS publishes annual PDFs, no API available"))

print(f"\n\u2705 Data loaded:")
print(f"  Crude imports: {len(df_imports)} months across {df_imports['duoarea'].nunique()} PADDs")
print(f"  Weekly stocks: {len(df_stocks)} observations")
print(f"  NatGas imports: {len(df_natgas)} months (Pipeline + LNG)")
print(f"  Helium: {len(df_helium)} years | STEO: {len(df_steo)} projections")

# --- Inventory data ---
from commodity_flow import inventory

if config.USE_LIVE_API and config.EIA_API_KEY:
    print("Fetching product-level stocks and consumption...")
    df_prod_stocks = inventory.fetch_product_stocks(config.EIA_API_KEY)
    df_prod_supplied = inventory.fetch_product_supplied(config.EIA_API_KEY)
    prov.record(DataSource("Product stocks", "EIA API v2", "petroleum/stoc/wstk", True,
        rows=len(df_prod_stocks), notes="By product: crude (SPR+commercial), gasoline, distillate, jet, propane"))
    prov.record(DataSource("Product supplied", "EIA API v2", "petroleum/cons/wpsup", True,
        rows=len(df_prod_supplied), notes="Weekly consumption proxy (MBBL/D)"))
else:
    print("Using offline inventory data")
    inv_data = inventory.generate_offline_inventory()
    df_prod_stocks = inv_data["stocks"]
    df_prod_supplied = inv_data["supplied"]
    prov.record(DataSource("Product stocks", "Offline", "offline generator", False,
        rows=len(df_prod_stocks)))
    prov.record(DataSource("Product supplied", "Offline", "offline generator", False,
        rows=len(df_prod_supplied)))

print(f"  Product stocks: {len(df_prod_stocks)} rows ({df_prod_stocks['product'].nunique()} products)")
print(f"  Product supplied: {len(df_prod_supplied)} rows (consumption proxy)")

---
### Data Provenance

All datasets and their sources, date ranges, and notes.

In [ ]:
from IPython.display import Markdown
display(Markdown(prov.summary()))

---
## Executive Summary

In [ ]:
# Compute headline metrics dynamically — all data loaded above
from commodity_flow import inventory as _inv

# Composite gap score
_sc = analysis.build_scorecard(df_imports, df_natgas, df_steo)
_actual = _sc[~_sc['is_forecast']]
_latest_z = _actual['composite_gap_score'].iloc[-1] if not _actual.empty else 0
_z_status = '🔴 ALERT' if _latest_z < -2 else '⚠️ Warning' if _latest_z < -1 else '✅ Normal'

# PADD 3 gap
_p3 = df_imports[df_imports['duoarea'] == 'PADD 3'].sort_values('date')
_p3_ma = _p3['value'].rolling(12, min_periods=12).mean().iloc[-1] if len(_p3) >= 12 else _p3['value'].mean()
_p3_latest = _p3['value'].iloc[-1]
_p3_gap = (_p3_latest - _p3_ma) / _p3_ma * 100

# Inventory metrics (already loaded)
_dos = _inv.compute_days_of_supply(df_prod_stocks, df_prod_supplied)
_gas_dos = _dos[_dos['product'] == 'EPM0']['days_of_supply'].iloc[-1] if 'EPM0' in _dos['product'].values else None
_dist_dos = _dos[_dos['product'] == 'EPD0']['days_of_supply'].iloc[-1] if 'EPD0' in _dos['product'].values else None
_spr_data = _inv.compute_spr_status(df_prod_stocks)
_spr_val = _spr_data.dropna().iloc[-1]['spr_mbbl'] / 1000 if not _spr_data.empty else None
_spr_pct = _spr_data.dropna().iloc[-1]['spr_pct'] if not _spr_data.empty else None

# NatGas status
_pipe = df_natgas[df_natgas['mode'] == 'Pipeline']['value_bcf']
_pipe_recent = _pipe.tail(3).mean()
_pipe_avg = _pipe.mean()
_ng_status = 'DISRUPTED' if _pipe_recent < _pipe_avg * 0.9 else 'Stable'

# STEO forward
_fwd = _sc[_sc['is_forecast']]
_fwd_avg = _fwd['composite_gap_score'].mean() if not _fwd.empty else 0
_fwd_dir = 'Tightening' if _fwd_avg < -1 else 'Normalizing' if _fwd_avg > 0 else 'Flat'

from IPython.display import Markdown
display(Markdown(f"""
| Metric | Current | Signal |
|--------|---------|--------|
| Composite Gap Score | {_latest_z:.1f}\u03c3 | {_z_status} |
| PADD 3 Import Trend | {_p3_gap:+.0f}% vs 12mo avg | {'🔴 Gap detected' if _p3_gap < -10 else '⚠️ Below trend' if _p3_gap < -5 else '✅ Normal'} |
| Gasoline Days of Supply | {f"{_gas_dos:.0f} days" if _gas_dos else "N/A"} | {'🔴 Tight' if _gas_dos and _gas_dos < 25 else '✅ Adequate' if _gas_dos else '—'} |
| Distillate Days of Supply | {f"{_dist_dos:.0f} days" if _dist_dos else "N/A"} | {'🔴 Tight' if _dist_dos and _dist_dos < 30 else '✅ Adequate' if _dist_dos else '—'} |
| Crude SPR | {f"{_spr_val:.0f}M bbl ({_spr_pct:.0f}%)" if _spr_val else "N/A"} | {'⚠️ Below 2020 levels' if _spr_pct and _spr_pct < 50 else '✅ Normal' if _spr_pct else '—'} |
| NatGas Pipeline Imports | {_ng_status} | {'🔴 Disrupted' if _ng_status == 'DISRUPTED' else '✅ Normal'} |
| STEO Forward View | {_fwd_avg:.1f}\u03c3 ({_fwd_dir}) | {'⚠️ Tightening' if _fwd_dir == 'Tightening' else '✅ ' + _fwd_dir} |

> **Bottom line:** {'Physical supply signals are elevated. Delivery gaps detected with composite score in warning territory.' if _latest_z < -1 else 'Physical supply signals are within normal ranges. No immediate delivery gap concerns.'}
"""))

---
## 2. Crude Oil Import Gaps by PADD

**Method:** Compare trailing 12-month average to recent months. A "gap" = actual imports below the trailing norm by > 1 std dev.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=True)
axes = axes.flatten()

gap_summary = []

for i, (padd, name) in enumerate(config.PADDS.items()):
    ax = axes[i]
    sub = df_imports[df_imports["duoarea"] == padd].sort_values("date").copy()
    sub = analysis.detect_gaps(sub, column="value", window=12, threshold=-1.0)

    # Compute rolling stats on FULL series for stable estimates
    sub["ma12"] = sub["value"].rolling(12, min_periods=12).mean()
    sub["std12"] = sub["value"].rolling(12, min_periods=12).std()
    sub["upper_band"] = sub["ma12"] + sub["std12"]
    sub["lower_band"] = sub["ma12"] - sub["std12"]

    # Trim to display window: start where 12-mo avg is valid
    plot_data = sub.dropna(subset=["ma12"])

    ax.plot(plot_data["date"], plot_data["value"], 'o-', ms=3, label="Actual imports", color="#2563eb")
    ax.plot(plot_data["date"], plot_data["ma12"], '--', label="12-mo avg", color="#64748b", lw=2)
    ax.fill_between(plot_data["date"], plot_data["lower_band"], plot_data["upper_band"],
                     alpha=0.12, color="#64748b", label="\u00b11\u03c3 band")

    # Highlight gap periods
    for gd in plot_data[plot_data["in_gap"]]["date"]:
        ax.axvspan(gd - timedelta(days=15), gd + timedelta(days=15),
                   alpha=0.25, color="#ef4444")

    ax.set_title(f"{padd}: {name}", fontweight="bold")
    ax.set_ylabel("MBBL (Thousand Barrels)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    ax.legend(fontsize=8, loc="lower left")

    # Summary stats
    recent = plot_data[plot_data["date"] >= "2025-09-01"]
    if recent["in_gap"].any():
        gap_months = recent[recent["in_gap"]]["date"].dt.strftime("%Y-%m").tolist()
        avg_deficit = recent[recent["in_gap"]]["z_score"].mean()
        gap_summary.append({
            "padd": padd, "region": name,
            "gap_months": ", ".join(gap_months),
            "avg_z_score": round(avg_deficit, 2),
        })

axes[5].set_visible(False)
fig.suptitle("Crude Oil Imports by PADD \u2014 Delivery Gap Detection",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

if gap_summary:
    print("\n=== DETECTED DELIVERY GAPS (recent) ===")
    display(pd.DataFrame(gap_summary))
else:
    print("No significant delivery gaps detected in recent window.")

In [ ]:
# Signal callout
_padds_in_gap = [s['padd'] for s in gap_summary] if gap_summary else []
if _padds_in_gap:
    print(f'\u26a0\ufe0f SIGNAL: Delivery gaps detected in {", ".join(_padds_in_gap)}.')
    print(f'  {len(_padds_in_gap)} of 5 PADDs showing imports below 12-month trailing average.')
    print(f'  Watch for stock draws to accelerate in affected regions.')
else:
    print('\u2705 No significant delivery gaps detected. All PADDs within normal import ranges.')

---
## 3. Inventory Adequacy

Physical stock levels, consumption rates, and strategic reserves — are there enough barrels on hand?

### Weekly Stocks Drawdown

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=False)
axes = axes.flatten()

for i, (padd, name) in enumerate(config.PADDS.items()):
    ax = axes[i]
    sub = df_stocks[df_stocks["duoarea"] == padd].sort_values("date").copy()
    sub["delta_4w"] = sub["value"].diff(4)

    color = sub["delta_4w"].apply(lambda x: "#ef4444" if x < 0 else "#22c55e")
    ax.bar(sub["date"], sub["delta_4w"], width=5, color=color, alpha=0.7)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"{padd}: {name}\n4-Week Stock Change", fontweight="bold")
    ax.set_ylabel("MBBL change")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis='x', rotation=45)

axes[5].set_visible(False)
fig.suptitle("Weekly Crude Stock Changes by PADD \u2014 Drawdown Alert",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### Days of Supply

Stocks / daily consumption = days of coverage at current draw rates.  
Below ~25 days for gasoline or ~30 days for distillate signals tight supply.

In [ ]:
df_dos = inventory.compute_days_of_supply(df_prod_stocks, df_prod_supplied)

if not df_dos.empty:
    products_with_dos = sorted(df_dos["product"].unique())
    n = len(products_with_dos)
    fig, axes = plt.subplots(1, min(n, 4), figsize=(5 * min(n, 4), 5), sharey=False)
    if n == 1:
        axes = [axes]

    colors = {"EPM0F": "#22c55e", "EPD0": "#2563eb", "EPJK": "#7c3aed", "EPLLPZ": "#f97316"}

    for ax, prod in zip(axes, products_with_dos[:4]):
        sub = df_dos[df_dos["product"] == prod].sort_values("date")
        name = sub["product_name"].iloc[0] if "product_name" in sub.columns else prod
        c = colors.get(prod, "#64748b")
        ax.plot(sub["date"], sub["days_of_supply"], color=c, lw=1.5)
        ax.set_title(name, fontweight="bold")
        ax.set_ylabel("Days of Supply")
        ax.tick_params(axis='x', rotation=30)
        # Reference lines
        median_dos = sub["days_of_supply"].median()
        ax.axhline(median_dos, color="#64748b", ls="--", lw=0.8, alpha=0.5)

    fig.suptitle("Days of Supply by Product \u2014 Stocks / Daily Consumption",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Days-of-supply data not available (need both stocks and consumption data).")

### Stocks vs 5-Year Seasonal Average

The market benchmark: current stocks compared to the trailing 5-year average for the same week.  
**Above average** = bearish for prices. **Below average** = bullish signal.

In [ ]:
df_seasonal = inventory.compute_seasonal_comparison(df_prod_stocks)

if not df_seasonal.empty:
    products_seasonal = sorted(df_seasonal["product"].unique())
    n = len(products_seasonal)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, prod in enumerate(products_seasonal[:6]):
        ax = axes[i]
        sub = df_seasonal[df_seasonal["product"] == prod].sort_values("date")
        name = sub["product_name"].iloc[0] if not sub.empty else prod

        ax.fill_between(sub["date"], sub["min_5yr"], sub["max_5yr"],
                         alpha=0.15, color="#64748b", label="5yr range")
        ax.plot(sub["date"], sub["avg_5yr"], '--', color="#64748b", lw=1.5, label="5yr avg")
        ax.plot(sub["date"], sub["current"], color="#dc2626", lw=2, label="Current")
        ax.set_title(name, fontweight="bold")
        ax.set_ylabel("MBBL")
        ax.legend(fontsize=8)
        ax.tick_params(axis='x', rotation=30)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Current Stocks vs 5-Year Seasonal Range",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # Deviation summary table
    latest = df_seasonal.sort_values("date").groupby("product").last().reset_index()
    if not latest.empty:
        print("\n=== Latest Week: Deviation from 5-Year Average ===")
        display(latest[["product_name", "current", "avg_5yr", "deviation_pct", "deviation_sigma"]].round(1))
else:
    print("Seasonal comparison not available (need multiple years of data).")

### SPR vs Commercial Crude Stocks

The Strategic Petroleum Reserve has been drawn down significantly since 2022.  
Lower SPR = less buffer for supply shocks.

In [ ]:
df_spr = inventory.compute_spr_status(df_prod_stocks)

if not df_spr.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    ax1.stackplot(df_spr["date"],
                  df_spr["spr_mbbl"].fillna(0),
                  df_spr["commercial_mbbl"].fillna(0),
                  labels=["SPR", "Commercial"],
                  colors=["#f97316", "#2563eb"], alpha=0.7)
    ax1.set_title("US Crude Oil Stocks \u2014 SPR vs Commercial", fontweight="bold")
    ax1.set_ylabel("Thousand Barrels")
    ax1.legend(loc="upper right")

    ax2.plot(df_spr["date"], df_spr["spr_pct"], color="#f97316", lw=2)
    ax2.fill_between(df_spr["date"], df_spr["spr_pct"], alpha=0.15, color="#f97316")
    ax2.set_title("SPR as % of Total Crude Stocks", fontweight="bold")
    ax2.set_ylabel("SPR %")
    ax2.axhline(50, color="#64748b", ls="--", lw=0.8, label="50% line")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    latest_spr = df_spr.dropna().iloc[-1]
    print(f"\nLatest: SPR {latest_spr['spr_mbbl']:,.0f} MBBL | "
          f"Commercial {latest_spr['commercial_mbbl']:,.0f} MBBL | "
          f"SPR share: {latest_spr['spr_pct']:.1f}%")
else:
    print("SPR data not available.")

---
## 4. Natural Gas Imports — Helium Supply Leading Indicator

Helium is extracted during natural gas processing (cryogenic separation). No NatGas throughput → no helium production. Pipeline + LNG disruptions cascade to helium supply.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for mode, color, ax in [("Pipeline", "#2563eb", ax1), ("LNG", "#7c3aed", ax2)]:
    sub = df_natgas[df_natgas["mode"] == mode].sort_values("date")
    sub["ma6"] = sub["value_bcf"].rolling(6, min_periods=3).mean()
    sub["std6"] = sub["value_bcf"].rolling(6, min_periods=3).std()

    ax.plot(sub["date"], sub["value_bcf"], 'o-', ms=3, color=color, label=f"{mode} imports")
    ax.plot(sub["date"], sub["ma6"], '--', color="#64748b", label="6-mo avg")
    ax.fill_between(sub["date"], sub["ma6"] - sub["std6"], sub["ma6"] + sub["std6"],
                     alpha=0.1, color="#64748b")

    gap_mask = sub["value_bcf"] < (sub["ma6"] - 1.5 * sub["std6"])
    ax.scatter(sub[gap_mask]["date"], sub[gap_mask]["value_bcf"],
               color="#ef4444", s=80, zorder=5, label="Disruption", marker="v")

    ax.set_title(f"Natural Gas Imports \u2014 {mode}", fontweight="bold")
    ax.set_ylabel("Billion Cubic Feet (Bcf)")
    ax.legend(fontsize=9)

fig.suptitle("Natural Gas Import Disruptions \u2192 Helium Supply Chain Risk",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Helium Supply-Demand Gap + Price Signal

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = df_helium["year"]
ax1.bar(x - 0.2, df_helium["world_production_Mcm"], 0.35, label="World Production", color="#2563eb")
ax1.bar(x + 0.2, df_helium["world_demand_Mcm"], 0.35, label="World Demand", color="#f97316")
ax1.axhline(0, color="black", lw=0.5)

for _, row in df_helium[df_helium["supply_gap_Mcm"] < 0].iterrows():
    ax1.axvspan(row["year"] - 0.45, row["year"] + 0.45, alpha=0.15, color="#ef4444")

ax1.set_title("Global Helium Supply vs. Demand", fontweight="bold")
ax1.set_ylabel("Million Cubic Meters")
ax1.legend()
ax1.set_xticks(x)

ax2.plot(x, df_helium["blm_price_usd_per_mcf"], 'o-', color="#dc2626", lw=2, ms=7)
ax2.fill_between(x, df_helium["blm_price_usd_per_mcf"], alpha=0.1, color="#dc2626")
ax2.set_title("Helium Price (BLM Grade-A)", fontweight="bold")
ax2.set_ylabel("USD per Mcf")
ax2.set_xticks(x)

ax2.annotate("Shortage pricing\nkicks in", xy=(2023, 35),
             xytext=(2021, 38), fontsize=10,
             arrowprops=dict(arrowstyle="->", color="#dc2626"),
             color="#dc2626", fontweight="bold")

fig.suptitle("Helium: No Substitutes, Constrained Supply, Surging Demand (MRI, Semiconductors, Quantum)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\n=== Helium Supply Gap Summary ===")
display(df_helium[["year", "supply_gap_Mcm", "blm_price_usd_per_mcf"]].tail(5))

---
## 6. Composite Gap Scorecard + STEO Forward Overlay

Combines oil + natural gas z-scores into a single delivery-risk metric.  
STEO forward projections extend the scorecard into forecast territory (dashed).

In [ ]:
scorecard = analysis.build_scorecard(df_imports, df_natgas, df_steo)

fig, ax = plt.subplots(figsize=(16, 6))

# Split actual vs forecast
actual = scorecard[~scorecard["is_forecast"]]
forecast = scorecard[scorecard["is_forecast"]]

ax.plot(actual.index, actual["oil_import_z"], label="Oil Import Z", color="#2563eb", lw=1.5)
ax.plot(actual.index, actual["natgas_import_z"], label="NatGas Import Z", color="#7c3aed", lw=1.5)
ax.plot(actual.index, actual["composite_gap_score"], label="Composite Gap Score",
        color="#dc2626", lw=2.5)

# STEO forecast extension
if not forecast.empty:
    # Connect actual to forecast
    bridge = pd.concat([actual.tail(1), forecast])
    ax.plot(bridge.index, bridge["composite_gap_score"], '--',
            color="#dc2626", lw=1.5, alpha=0.6, label="STEO Forecast")
    ax.axvspan(forecast.index.min(), forecast.index.max(),
               alpha=0.05, color="#fbbf24", label="Forecast zone")

ax.axhline(-1, color="#f97316", ls="--", label="Warning (-1\u03c3)")
ax.axhline(-2, color="#ef4444", ls="--", label="Critical (-2\u03c3)")
ax.axhline(0, color="black", lw=0.5)

ax.fill_between(actual.index, -2, actual["composite_gap_score"],
                where=actual["composite_gap_score"] < -2,
                alpha=0.3, color="#ef4444", interpolate=True)

ax.set_title("Composite Delivery Gap Scorecard \u2014 Oil + Natural Gas (Helium Proxy) + STEO Forward",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Z-Score (negative = below normal)")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

# Alert table
alerts = actual[actual["composite_gap_score"] < -1].tail(10)
if not alerts.empty:
    print("\n=== MONTHS WITH GAP ALERTS (composite < -1\u03c3) ===")
    display(alerts.round(2))

---
## 7. Futures Price Overlay — Physical vs. Market Divergence

Compare physical delivery gap z-scores with commodity futures price z-scores.  
**Divergence** = physical supply is tighter/looser than the market is pricing.

In [ ]:
try:
    df_fut = futures.fetch_futures_curves()
    df_fut = futures.compute_futures_z_scores(df_fut)
    has_futures = not df_fut.empty
except Exception as e:
    print(f"Futures data unavailable: {e}")
    has_futures = False

if has_futures:
    symbols = df_fut["symbol"].unique()
    fig, axes = plt.subplots(1, len(symbols), figsize=(6 * len(symbols), 5), sharey=True)
    if len(symbols) == 1:
        axes = [axes]

    colors = {"CL=F": "#2563eb", "NG=F": "#7c3aed", "RB=F": "#16a34a", "HO=F": "#ea580c"}

    for ax, sym in zip(axes, symbols):
        sub = df_fut[df_fut["symbol"] == sym].sort_values("date")
        name = sub["name"].iloc[0]
        c = colors.get(sym, "#64748b")

        ax.plot(sub["date"], sub["futures_z"], color=c, lw=1.5, label=f"{name} Z")
        ax.axhline(-1, color="#f97316", ls="--", lw=0.8)
        ax.axhline(1, color="#22c55e", ls="--", lw=0.8)
        ax.axhline(0, color="black", lw=0.5)
        ax.fill_between(sub["date"], -1, sub["futures_z"],
                        where=sub["futures_z"] < -1, alpha=0.2, color="#ef4444", interpolate=True)
        ax.fill_between(sub["date"], 1, sub["futures_z"],
                        where=sub["futures_z"] > 1, alpha=0.2, color="#22c55e", interpolate=True)
        ax.set_title(f"{name} ({sym})", fontweight="bold")
        ax.set_ylabel("Price Z-Score" if ax == axes[0] else "")
        ax.tick_params(axis='x', rotation=30)

    fig.suptitle("Commodity Futures Z-Scores \u2014 Compare with Physical Gap Scorecard",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping futures overlay (no data available).")

---
## Key Takeaways

In [ ]:
from IPython.display import Markdown as _Md

_takeaways = []

# 1. Import gaps
if gap_summary:
    regions = ', '.join([s['region'] for s in gap_summary])
    _takeaways.append(f"**Import gaps widening** in {regions} — crude deliveries below 12-month norm. Watch for stock draws to accelerate.")
else:
    _takeaways.append("**Import gaps:** All PADDs within normal import ranges. No immediate concern.")

# 2. Inventory
try:
    if isinstance(_gas_dos, float):
        status_word = "adequate" if _gas_dos >= 25 else "**TIGHT**"
        _takeaways.append(f"**Gasoline inventory** at {_gas_dos:.0f} days of supply ({status_word}). Distillate and jet fuel should be monitored for seasonal draws.")
except NameError:
    _takeaways.append("**Inventory:** See Section 3b for detailed product-level analysis.")

# 3. NatGas / Helium
_pipe = df_natgas[df_natgas['mode'] == 'Pipeline']['value_bcf']
_pipe_recent = _pipe.tail(3).mean()
_pipe_avg = _pipe.mean()
_ng_status = "DISRUPTED" if _pipe_recent < _pipe_avg * 0.9 else "stable"
_takeaways.append(f"**NatGas pipeline imports {_ng_status}.** LNG imports near zero (US is net exporter). Any pipeline disruption has no LNG buffer — helium co-production at risk.")

# 4. Scorecard
if _latest_z < -1:
    _takeaways.append(f"**Composite scorecard at {_latest_z:.1f}\u03c3** — supply tighter than trailing norms. Physical delivery risk elevated.")
else:
    _takeaways.append(f"**Composite scorecard at {_latest_z:.1f}\u03c3** — within normal range.")

# 5. Forward view
_fwd = _sc[_sc['is_forecast']]
if not _fwd.empty:
    _fwd_avg = _fwd['composite_gap_score'].mean()
    direction = "continued tightness" if _fwd_avg < -1 else "normalization"
    _takeaways.append(f"**STEO forward view:** {_fwd_avg:.1f}\u03c3 implied gap score. EIA projections suggest {direction} ahead.")

_md_text = "\n".join(f"{j+1}. {t}" for j, t in enumerate(_takeaways))
_md_text += "\n\n*For trade ideas and scenario analysis, see the [Market Intelligence Briefing](03_market_intelligence_briefing.ipynb).*"
display(_Md(_md_text))


---
## Appendix

Reference material and supplementary data sources.

### A. AIS Tanker Tracker

Connects to AISstream.io websocket to collect real-time tanker positions near US ports.
**Requires:** AISstream.io API key in `.env`. Enhancement planned for deeper integration with PADD gap analysis.

In [ ]:
from commodity_flow import ais
import asyncio

if config.AISSTREAM_API_KEY:
    print("Connecting to AISstream.io (60s collection window)...")
    df_tankers = await ais.track_tankers(
        config.AISSTREAM_API_KEY,
        duration_seconds=60,
        max_positions=200,
    )
    print(f"Collected {len(df_tankers)} tanker positions.")

    if not df_tankers.empty:
        fig, ax = plt.subplots(figsize=(14, 8))
        scatter = ax.scatter(
            df_tankers["lon"], df_tankers["lat"],
            c=df_tankers["speed_kn"], cmap="RdYlGn_r",
            s=20, alpha=0.7, edgecolors="none"
        )
        plt.colorbar(scatter, label="Speed (knots)")

        # Draw bounding boxes
        for box in ais.US_PORT_BOXES:
            sw, ne = box
            ax.plot([sw[1], ne[1], ne[1], sw[1], sw[1]],
                    [sw[0], sw[0], ne[0], ne[0], sw[0]],
                    'b--', alpha=0.3, lw=1)

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_title(f"Tanker Positions ({len(df_tankers)} reports)", fontweight="bold")
        plt.tight_layout()
        plt.show()

        display(df_tankers.head(10))
else:
    print("AISstream.io API key not set. Set AISSTREAM_API_KEY in .env to enable.")
    print("Register free at: https://aisstream.io")

### B. Derivative Commodity Map

Commodities whose supply chains are measurable through physical flow data.

In [ ]:
commodity_map = pd.DataFrame([
    {"commodity": "Crude Oil",
     "instruments": "WTI/Brent futures, crack spreads, PADD basis",
     "data_source": "EIA API v2 (petroleum/move/imp, crude-oil-imports)",
     "shipping_trackable": True,
     "lead_time": "2-6 weeks (AIS tanker ETA)",
     "gap_signal": "PADD import shortfall vs 12-mo avg"},
    {"commodity": "Helium",
     "instruments": "No direct futures; equities (DME, RLPI, PHX)",
     "data_source": "USGS MCS (annual); BLM auction prices",
     "shipping_trackable": False,
     "lead_time": "Lagging (annual); proxy via NatGas throughput",
     "gap_signal": "NatGas disruptions \u2192 helium co-production collapse"},
    {"commodity": "Natural Gas (LNG)",
     "instruments": "Henry Hub, JKM (Asia LNG), TTF (Europe)",
     "data_source": "EIA API v2 (natural-gas/move), FERC LNG reports",
     "shipping_trackable": True,
     "lead_time": "2-4 weeks (LNG carrier AIS)",
     "gap_signal": "LNG import volume vs seasonal norm"},
    {"commodity": "Gasoline / Distillates",
     "instruments": "RBOB futures, heating oil futures, crack spreads",
     "data_source": "EIA API v2 (petroleum/move/wkly, petroleum/stoc/wstk)",
     "shipping_trackable": True,
     "lead_time": "1-3 weeks (product tanker AIS)",
     "gap_signal": "Weekly stocks drawdown rate by PADD"},
    {"commodity": "Propane / NGLs",
     "instruments": "Mt. Belvieu propane, ethane/butane swaps",
     "data_source": "EIA API v2 (petroleum/stoc, natural-gas/move)",
     "shipping_trackable": True,
     "lead_time": "1-2 weeks",
     "gap_signal": "NGL stock levels + fractionation utilization"},
    {"commodity": "Lithium",
     "instruments": "CME lithium hydroxide futures, equity (ALB, SQM)",
     "data_source": "USGS MCS, Fastmarkets, Benchmark Minerals",
     "shipping_trackable": True,
     "lead_time": "4-8 weeks (bulk carrier AIS from Chile/Australia)",
     "gap_signal": "Port departures vs contract volumes"},
])

display(commodity_map.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold')]}
]))